# Model Comparison

End-to-end pipeline: loads raw data, runs the **v8 leakage-free preprocessing**, trains four single-model baselines **AND** the tier-specific cascade classifier (confidence margin = 0.10), then prints a comparison table and chart.

**Models:** Logistic Regression · XGBoost · Random Forest · GBM · Cascade Classifier

---
## 1 — Imports & Configuration

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, warnings, itertools, time
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.feature_selection import mutual_info_classif
from category_encoders import TargetEncoder
import xgboost as xgb, lightgbm as lgb

RANDOM_STATE = 42
MARGIN       = 0.10
N_FOLDS      = 5
DATA_PATH    = 'DataCo_Weather_Encoded.csv'
TARGET_COL   = 'Late_delivery_risk'
TEST_SIZE    = 0.2
np.random.seed(RANDOM_STATE)
print('Imports & config ✅')

---
## 2 — Preprocessing (v8 Pipeline)
Full leakage-free pipeline: Load → Clean → Feature Engineering → Split → Impute → Outlier Treatment → Geo Feature → Log Transform → Encode → Normalize → Feature Reduction

### 2.1 — Load & Clean + Feature Engineering

In [ ]:
# ── Load raw data ──
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Raw: {df.shape}')
y = df[TARGET_COL].copy()
df.drop(columns=[TARGET_COL], inplace=True)

# ── Drop leakage / useless / duplicate columns ──
drop_cols = [
    'Days for shipping (real)', 'shipping date (DateOrders)', 'shipping_date',
    'Target_Date', 'Delivery Status', 'Order Status',
    'Customer Email', 'Customer Password', 'Product Description',
    'Product Status', 'Product Image',
    'Order Item Total', 'Product Price', 'Product Category Id',
    'lon_round', 'lat_round',
]
df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)
print(f'After drop: {df.shape}')

# ── Save Shipping Mode before engineering ──
shipping_mode_series = df['Shipping Mode'].copy()

# ── Feature Engineering (row-level only — no global stats) ──
date_col = 'order date (DateOrders)'
if date_col in df.columns:
    dt = pd.to_datetime(df[date_col], errors='coerce')
    df['order_month']      = dt.dt.month
    df['order_dayofweek']  = dt.dt.dayofweek
    df['month_sin'] = np.sin(2 * np.pi * df['order_month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['order_month'] / 12)
    df['dow_sin']   = np.sin(2 * np.pi * df['order_dayofweek'] / 7)
    df['dow_cos']   = np.cos(2 * np.pi * df['order_dayofweek'] / 7)
    df['is_weekend'] = (df['order_dayofweek'] >= 5).astype(int)
    df.drop(columns=[date_col], inplace=True)

if 'Shipping Mode' in df.columns:
    df['ship_priority'] = df['Shipping Mode'].map(
        {'Standard Class': 1, 'Second Class': 2, 'First Class': 3, 'Same Day': 4}).fillna(1)

if {'Days for shipment (scheduled)', 'ship_priority'}.issubset(df.columns):
    df['sched_x_priority'] = df['Days for shipment (scheduled)'] * df['ship_priority']

if {'Sales', 'Order Item Quantity'}.issubset(df.columns):
    df['price_per_item'] = df['Sales'] / (df['Order Item Quantity'] + 1)
if {'Benefit per order', 'Sales'}.issubset(df.columns):
    df['profit_margin'] = df['Benefit per order'] / (df['Sales'] + 1)
if {'Order Item Discount', 'Order Item Product Price'}.issubset(df.columns):
    df['discount_ratio'] = df['Order Item Discount'] / (df['Order Item Product Price'] + 1)

print(f'After engineering: {df.shape}')

### 2.2 — Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

# Split shipping mode with identical indices
_, _, sm_train, sm_test = train_test_split(
    shipping_mode_series, shipping_mode_series,
    test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)
sm_train = sm_train.reset_index(drop=True)
sm_test  = sm_test.reset_index(drop=True)

print(f'Train: {X_train.shape}  Test: {X_test.shape}')

### 2.3 — Impute → Outliers → Geo → Log → Encode → Normalize

In [ ]:
# ── Imputation (train-fit) ──
for col in X_train.select_dtypes('number').columns:
    if X_train[col].isna().any() or X_test[col].isna().any():
        fill = X_train[col].mean()
        X_train[col] = X_train[col].fillna(fill)
        X_test[col]  = X_test[col].fillna(fill)
for col in X_train.select_dtypes('object').columns:
    if X_train[col].isna().any() or X_test[col].isna().any():
        mode = X_train[col].mode()
        fill = mode[0] if len(mode) > 0 else 'Unknown'
        X_train[col] = X_train[col].fillna(fill)
        X_test[col]  = X_test[col].fillna(fill)
print(f'Imputation done — NaNs: train={X_train.isna().sum().sum()}, test={X_test.isna().sum().sum()}')

# ── Outlier treatment (IQR bounds from train) ──
treated = 0
for col in X_train.select_dtypes('number').columns:
    Q1, Q3 = X_train[col].quantile(0.25), X_train[col].quantile(0.75)
    IQR = Q3 - Q1
    if IQR == 0: continue
    lo, hi = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    if ((X_train[col] < lo) | (X_train[col] > hi)).any():
        X_train[col] = X_train[col].clip(lo, hi)
        X_test[col]  = X_test[col].clip(lo, hi)
        treated += 1
print(f'Outlier capping: {treated} columns')

# ── Geographic feature (dist_center, median from train) ──
if {'Latitude', 'Longitude'}.issubset(X_train.columns):
    lat_med, lon_med = X_train['Latitude'].median(), X_train['Longitude'].median()
    X_train['dist_center'] = np.sqrt((X_train['Latitude']-lat_med)**2 + (X_train['Longitude']-lon_med)**2)
    X_test['dist_center']  = np.sqrt((X_test['Latitude']-lat_med)**2  + (X_test['Longitude']-lon_med)**2)
    print('Added dist_center')

# ── Log transform (skewness from train) ──
log_candidates = ['Sales', 'Order Item Discount', 'Order Item Product Price',
                  'Benefit per order', 'Order Item Profit Ratio',
                  'dist_center', 'price_per_item', 'Sales per customer',
                  'Order Profit Per Order']
n_log = 0
for col in log_candidates:
    if col in X_train.columns and abs(X_train[col].skew()) > 0.5:
        X_train[f'{col}_log'] = np.log1p(X_train[col].clip(lower=0))
        X_test[f'{col}_log']  = np.log1p(X_test[col].clip(lower=0))
        n_log += 1
print(f'Log-transformed: {n_log} features')

# ── Encode categoricals (train-fit) ──
cat_cols = X_train.select_dtypes('object').columns.tolist()
low_card  = [c for c in cat_cols if X_train[c].nunique() < 5]
high_card = [c for c in cat_cols if X_train[c].nunique() >= 5]
if low_card:
    X_train = pd.get_dummies(X_train, columns=low_card, drop_first=True, dtype=int)
    X_test  = pd.get_dummies(X_test,  columns=low_card, drop_first=True, dtype=int)
    X_test  = X_test.reindex(columns=X_train.columns, fill_value=0)
if high_card:
    te = TargetEncoder(cols=high_card, smoothing=10)
    X_train[high_card] = te.fit_transform(X_train[high_card], y_train)
    X_test[high_card]  = te.transform(X_test[high_card])
print(f'Encoding: {len(low_card)} one-hot, {len(high_card)} target-encoded')

# ── MinMax normalization (train-fit) ──
scaler = MinMaxScaler()
cols = X_train.columns
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=cols, index=X_train.index)
X_test  = pd.DataFrame(scaler.transform(X_test),      columns=cols, index=X_test.index)
print(f'Normalized {len(cols)} features')

### 2.4 — Feature Reduction (Variance → Correlation → MI Top-30)

In [ ]:
initial_n = X_train.shape[1]

# Stage 1: near-zero variance
var = X_train.var()
low_var = var[var < 0.001].index.tolist()
X_train.drop(columns=low_var, inplace=True)
X_test.drop(columns=low_var, inplace=True)
print(f'[1] Dropped {len(low_var)} low-var → {X_train.shape[1]} remain')

# Stage 2: high correlation (|r| > 0.90)
corr = X_train.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
high_corr = [c for c in upper.columns if any(upper[c] > 0.90)]
X_train.drop(columns=high_corr, inplace=True)
X_test.drop(columns=high_corr, inplace=True)
print(f'[2] Dropped {len(high_corr)} high-corr → {X_train.shape[1]} remain')

# Safety: fill any residual NaN before MI (edge cases from encoding)
X_train = X_train.fillna(0)
X_test  = X_test.fillna(0)

# Stage 3: Mutual Information — keep top 30
TOP_N = 30
mi = pd.Series(
    mutual_info_classif(X_train, y_train, discrete_features='auto', random_state=RANDOM_STATE),
    index=X_train.columns
).sort_values(ascending=False)
selected = mi.head(TOP_N).index.tolist()
X_train = X_train[selected]
X_test  = X_test[selected]
print(f'[3] MI top-{TOP_N} selected → {X_train.shape[1]} final features (from {initial_n})')
print(f'\nTarget dist (train): Late={y_train.mean():.3f}  OnTime={1-y_train.mean():.3f}')

---
## 3 — Helper Functions (Confidence & Tuning)

In [ ]:
def predict_with_confidence(proba, threshold=0.5, margin=MARGIN):
    """Returns (predictions, confident_mask). At-Risk band: [threshold-margin, threshold+margin)"""
    pred = (proba >= threshold).astype(int)
    conf = (proba < threshold - margin) | (proba >= threshold + margin)
    return pred, conf

def evaluate_with_confidence(y_true, proba, threshold=0.5, margin=MARGIN):
    """Compute standard and confidence-based accuracy."""
    pred, conf = predict_with_confidence(proba, threshold, margin)
    acc_all  = accuracy_score(y_true, pred)
    n_conf   = conf.sum()
    n_risk   = (~conf).sum()
    acc_conf = accuracy_score(y_true[conf], pred[conf]) if n_conf > 0 else float('nan')
    return {'acc_all': acc_all, 'acc_conf': acc_conf, 'n_total': len(y_true),
            'n_conf': n_conf, 'n_risk': n_risk, 'pct_risk': n_risk / len(y_true) * 100}

def tune_model(estimator_cls, param_grid, X_tr, y_tr, fixed_kwargs=None, n_folds=N_FOLDS):
    """Grid-search via StratifiedKFold CV. Returns (model, best_params, best_cv_acc)."""
    fixed_kwargs = fixed_kwargs or {}
    kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    best_acc, best_params = 0.0, {}
    keys, values = list(param_grid.keys()), list(param_grid.values())
    for combo in itertools.product(*values):
        params = dict(zip(keys, combo))
        accs = []
        for tr, va in kf.split(X_tr, y_tr):
            m = estimator_cls(**params, **fixed_kwargs)
            m.fit(X_tr.iloc[tr], y_tr.iloc[tr])
            accs.append(accuracy_score(y_tr.iloc[va], m.predict(X_tr.iloc[va])))
        mean_a = float(np.mean(accs))
        if mean_a > best_acc:
            best_acc, best_params = mean_a, params
    model = estimator_cls(**best_params, **fixed_kwargs)
    model.fit(X_tr, y_tr)
    return model, best_params, best_acc

print('Helpers defined ✅')

---
## 4 — Model Training (Single-Model Baselines)

### 4.1 — Logistic Regression

In [ ]:
results = {}

print('MODEL 1: LOGISTIC REGRESSION')
t0 = time.time()
lr_model, lr_params, lr_cv = tune_model(
    LogisticRegression, {'C': [0.01, 0.1, 1.0]}, X_train, y_train,
    fixed_kwargs=dict(max_iter=5000, solver='saga', penalty='l2',
                      random_state=RANDOM_STATE, n_jobs=-1))
lr_time = time.time() - t0
lr_proba = lr_model.predict_proba(X_test)[:, 1]
lr_eval  = evaluate_with_confidence(y_test.values, lr_proba)
print(f'  Best: {lr_params}  CV={lr_cv:.4f}  Time={lr_time:.1f}s')
print(f'  Std acc: {lr_eval["acc_all"]:.4f}  Conf acc: {lr_eval["acc_conf"]:.4f}  At Risk: {lr_eval["pct_risk"]:.1f}%')
results['Logistic Regression'] = {**lr_eval, 'cv': lr_cv, 'time': lr_time}

### 4.2 — XGBoost

In [ ]:
print('MODEL 2: XGBOOST')
t0 = time.time()
xgb_model, xgb_params, xgb_cv = tune_model(
    xgb.XGBClassifier,
    {'max_depth': [3,5,7], 'learning_rate': [0.05,0.1], 'n_estimators': [100,200]},
    X_train, y_train,
    fixed_kwargs=dict(eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1, verbosity=0))
xgb_time = time.time() - t0
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
xgb_eval  = evaluate_with_confidence(y_test.values, xgb_proba)
print(f'  Best: {xgb_params}  CV={xgb_cv:.4f}  Time={xgb_time:.1f}s')
print(f'  Std acc: {xgb_eval["acc_all"]:.4f}  Conf acc: {xgb_eval["acc_conf"]:.4f}  At Risk: {xgb_eval["pct_risk"]:.1f}%')
results['XGBoost'] = {**xgb_eval, 'cv': xgb_cv, 'time': xgb_time}

### 4.3 — Random Forest

In [ ]:
print('MODEL 3: RANDOM FOREST')
t0 = time.time()
rf_model, rf_params, rf_cv = tune_model(
    RandomForestClassifier,
    {'max_depth': [8,12], 'n_estimators': [100,200]},
    X_train, y_train,
    fixed_kwargs=dict(random_state=RANDOM_STATE, n_jobs=-1))
rf_time = time.time() - t0
rf_proba = rf_model.predict_proba(X_test)[:, 1]
rf_eval  = evaluate_with_confidence(y_test.values, rf_proba)
print(f'  Best: {rf_params}  CV={rf_cv:.4f}  Time={rf_time:.1f}s')
print(f'  Std acc: {rf_eval["acc_all"]:.4f}  Conf acc: {rf_eval["acc_conf"]:.4f}  At Risk: {rf_eval["pct_risk"]:.1f}%')
results['Random Forest'] = {**rf_eval, 'cv': rf_cv, 'time': rf_time}

### 4.4 — Gradient Boosting Machine

In [ ]:
print('MODEL 4: GRADIENT BOOSTING MACHINE')
t0 = time.time()
gbm_params = {'max_depth': 3, 'learning_rate': 0.1, 'n_estimators': 150}
print(f'  Fixed params (sklearn GBC single-threaded): {gbm_params}')

kf_gbm = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
gbm_cv_accs = []
for tr, va in kf_gbm.split(X_train, y_train):
    m = GradientBoostingClassifier(**gbm_params, random_state=RANDOM_STATE)
    m.fit(X_train.iloc[tr], y_train.iloc[tr])
    gbm_cv_accs.append(accuracy_score(y_train.iloc[va], m.predict(X_train.iloc[va])))
    print(f'    Fold done: acc={gbm_cv_accs[-1]:.4f}')
gbm_cv = float(np.mean(gbm_cv_accs))

gbm_model = GradientBoostingClassifier(**gbm_params, random_state=RANDOM_STATE)
gbm_model.fit(X_train, y_train)
gbm_time = time.time() - t0
gbm_proba = gbm_model.predict_proba(X_test)[:, 1]
gbm_eval  = evaluate_with_confidence(y_test.values, gbm_proba)
print(f'  CV={gbm_cv:.4f}  Time={gbm_time:.1f}s')
print(f'  Std acc: {gbm_eval["acc_all"]:.4f}  Conf acc: {gbm_eval["acc_conf"]:.4f}  At Risk: {gbm_eval["pct_risk"]:.1f}%')
results['GBM'] = {**gbm_eval, 'cv': gbm_cv, 'time': gbm_time}

---
## 5 — Combined Model (Cascade Classifier)

### 5.1 — Standard Class LightGBM Helper

In [ ]:
def train_standard_class_lgbm(X_tr, y_tr):
    """LightGBM with is_unbalance + OOF threshold tuning. Returns (model, best_threshold)."""
    fixed = dict(is_unbalance=True, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
    model, best_params, best_acc = tune_model(
        lgb.LGBMClassifier,
        {'num_leaves': [31,63,127], 'learning_rate': [0.05,0.1], 'n_estimators': [200,300]},
        X_tr, y_tr, fixed_kwargs=fixed)
    print(f'     LGBM: {best_params}  CV={best_acc:.4f}')

    # OOF threshold sweep
    kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros(len(y_tr))
    for tr, va in kf.split(X_tr, y_tr):
        m = lgb.LGBMClassifier(**best_params, **fixed)
        m.fit(X_tr.iloc[tr], y_tr.iloc[tr])
        oof[va] = m.predict_proba(X_tr.iloc[va])[:, 1]

    y_arr = np.array(y_tr)
    best_thr, best_thr_acc = 0.5, 0.0
    for thr in np.arange(0.25, 0.52, 0.02):
        acc = accuracy_score(y_arr, (oof >= thr).astype(int))
        if acc > best_thr_acc:
            best_thr_acc, best_thr = acc, float(thr)
    print(f'     Best threshold: {best_thr:.2f}  (OOF acc={best_thr_acc:.4f})')
    return model, best_thr

print('LightGBM helper defined ✅')

### 5.2 — Run Full Cascade Classifier

In [ ]:
print('MODEL 5: CASCADE CLASSIFIER (tier-specific)')
t0 = time.time()

tiers = ['First Class', 'Second Class', 'Same Day', 'Standard Class']
all_preds  = np.full(len(y_test), -1)
all_conf   = np.zeros(len(y_test), dtype=bool)
all_forced = np.zeros(len(y_test), dtype=int)
tier_results = {}

# ── First Class: Logistic Regression ──
print('\n  ── First Class: Logistic Regression ──')
fc_tr, fc_te = (sm_train == 'First Class').values, (sm_test == 'First Class').values
fc_model, fc_p, fc_cv = tune_model(
    LogisticRegression, {'C': [0.001, 0.01, 0.1, 1.0, 10.0]},
    X_train[fc_tr], y_train[fc_tr],
    fixed_kwargs=dict(max_iter=5000, solver='saga', random_state=RANDOM_STATE, n_jobs=-1))
print(f'     C={fc_p["C"]}  CV={fc_cv:.4f}  n={fc_tr.sum()}')
fc_proba = fc_model.predict_proba(X_test[fc_te])[:, 1]
fc_pred, fc_conf = predict_with_confidence(fc_proba, 0.5, MARGIN)
all_preds[fc_te], all_conf[fc_te], all_forced[fc_te] = fc_pred, fc_conf, (fc_proba >= 0.5).astype(int)
tier_results['First Class'] = evaluate_with_confidence(y_test.values[fc_te], fc_proba, 0.5, MARGIN)

# ── Second Class: XGBoost ──
print('\n  ── Second Class: XGBoost ──')
sc_tr, sc_te = (sm_train == 'Second Class').values, (sm_test == 'Second Class').values
spw = float((y_train[sc_tr] == 1).sum() / max((y_train[sc_tr] == 0).sum(), 1))
sc_model, sc_p, sc_cv = tune_model(
    xgb.XGBClassifier,
    {'max_depth': [3,5,7], 'learning_rate': [0.05,0.1], 'n_estimators': [100,200]},
    X_train[sc_tr], y_train[sc_tr],
    fixed_kwargs=dict(scale_pos_weight=spw, eval_metric='logloss',
                      random_state=RANDOM_STATE, n_jobs=-1, verbosity=0))
print(f'     spw={spw:.2f}  {sc_p}  CV={sc_cv:.4f}  n={sc_tr.sum()}')
sc_proba = sc_model.predict_proba(X_test[sc_te])[:, 1]
sc_pred, sc_conf = predict_with_confidence(sc_proba, 0.5, MARGIN)
all_preds[sc_te], all_conf[sc_te], all_forced[sc_te] = sc_pred, sc_conf, (sc_proba >= 0.5).astype(int)
tier_results['Second Class'] = evaluate_with_confidence(y_test.values[sc_te], sc_proba, 0.5, MARGIN)

# ── Same Day: Gradient Boosting ──
print('\n  ── Same Day: GradientBoosting ──')
sd_tr, sd_te = (sm_train == 'Same Day').values, (sm_test == 'Same Day').values
sd_model, sd_p, sd_cv = tune_model(
    GradientBoostingClassifier,
    {'max_depth': [2,3,4], 'learning_rate': [0.1,0.2], 'n_estimators': [50,100,150]},
    X_train[sd_tr], y_train[sd_tr],
    fixed_kwargs=dict(random_state=RANDOM_STATE))
print(f'     {sd_p}  CV={sd_cv:.4f}  n={sd_tr.sum()}')
sd_proba = sd_model.predict_proba(X_test[sd_te])[:, 1]
sd_pred, sd_conf = predict_with_confidence(sd_proba, 0.5, MARGIN)
all_preds[sd_te], all_conf[sd_te], all_forced[sd_te] = sd_pred, sd_conf, (sd_proba >= 0.5).astype(int)
tier_results['Same Day'] = evaluate_with_confidence(y_test.values[sd_te], sd_proba, 0.5, MARGIN)

# ── Standard Class: LightGBM + threshold ──
print('\n  ── Standard Class: LightGBM + OOF threshold ──')
st_tr, st_te = (sm_train == 'Standard Class').values, (sm_test == 'Standard Class').values
st_model, st_thr = train_standard_class_lgbm(X_train[st_tr], y_train[st_tr])
st_proba = st_model.predict_proba(X_test[st_te])[:, 1]
st_pred, st_conf = predict_with_confidence(st_proba, st_thr, MARGIN)
all_preds[st_te], all_conf[st_te], all_forced[st_te] = st_pred, st_conf, (st_proba >= st_thr).astype(int)
tier_results['Standard Class'] = evaluate_with_confidence(y_test.values[st_te], st_proba, st_thr, MARGIN)

cascade_time = time.time() - t0

# ── Overall cascade metrics ──
all_preds[all_preds == -1] = 1  # force undecided
acc_all  = accuracy_score(y_test, all_forced)
n_conf   = all_conf.sum()
acc_conf = accuracy_score(y_test.values[all_conf], all_preds[all_conf]) if n_conf > 0 else 0.0
n_risk   = len(y_test) - n_conf
pct_risk = n_risk / len(y_test) * 100

print(f'\n  {"Tier":<18s}  {"N":>6}  {"AccAll":>7}  {"AccConf":>8}  {"AtRisk%":>8}')
print('  ' + '-' * 52)
for tier in tiers:
    r = tier_results[tier]
    print(f'  {tier:<18s}  {r["n_total"]:>6}  {r["acc_all"]:>7.4f}  {r["acc_conf"]:>8.4f}  {r["pct_risk"]:>7.1f}%')
print(f'\n  Cascade OVERALL: std={acc_all:.4f}  conf={acc_conf:.4f}  at_risk={pct_risk:.1f}%  time={cascade_time:.1f}s')

results['Cascade (ours)'] = {'acc_all': acc_all, 'acc_conf': acc_conf, 'n_total': len(y_test),
                              'n_conf': n_conf, 'n_risk': n_risk, 'pct_risk': pct_risk, 'time': cascade_time}

---
## 6 — Comparison Table & Visualization

### 6.1 — Comparison Table

In [ ]:
print('=' * 70)
print('FINAL COMPARISON TABLE  (confidence margin = ±0.10)')
print('=' * 70)
print(f'  {"Model":<22s}  {"StdAcc":>7}  {"ConfAcc":>8}  {"Improve":>8}  {"AtRisk%":>8}  {"Time(s)":>8}')
print('  ' + '-' * 68)
for name, r in results.items():
    imp = r['acc_conf'] - r['acc_all']
    print(f'  {name:<22s}  {r["acc_all"]:>7.4f}  {r["acc_conf"]:>8.4f}  {imp:>+8.4f}  {r["pct_risk"]:>7.1f}%  {r.get("time",0):>8.1f}')
best = max(results, key=lambda k: results[k]['acc_conf'])
print(f'\n  🏆 BEST CONFIDENCE ACCURACY: {best} ({results[best]["acc_conf"]:.4f})')

### 6.2 — Comparison Chart

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
names = list(results.keys())
colors = ['#4FC3F7', '#FF8A65', '#81C784', '#FFD54F', '#CE93D8']

# Standard vs Confidence Accuracy
ax = axes[0]
x, w = np.arange(len(names)), 0.35
ax.bar(x - w/2, [results[n]['acc_all'] for n in names], w, label='Standard', color='#90CAF9', edgecolor='white')
ax.bar(x + w/2, [results[n]['acc_conf'] for n in names], w, label='Confidence', color='#4FC3F7', edgecolor='white')
ax.set_ylabel('Accuracy'); ax.set_title('Standard vs Confidence Accuracy')
ax.set_xticks(x); ax.set_xticklabels(names, rotation=30, ha='right', fontsize=8)
ax.legend(); ax.set_ylim(0.70, 1.0); ax.grid(axis='y', alpha=0.3)

# At Risk %
ax = axes[1]
risk_vals = [results[n]['pct_risk'] for n in names]
bars = ax.bar(names, risk_vals, color=colors, edgecolor='white')
ax.set_ylabel('% At Risk'); ax.set_title('Percentage Flagged "At Risk"')
ax.set_xticklabels(names, rotation=30, ha='right', fontsize=8)
for bar, val in zip(bars, risk_vals):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3, f'{val:.1f}%', ha='center', va='bottom', fontsize=9)
ax.grid(axis='y', alpha=0.3)

# Improvement
ax = axes[2]
imps = [(results[n]['acc_conf'] - results[n]['acc_all']) * 100 for n in names]
ax.bar(names, imps, color=['#66BB6A' if v >= 0 else '#EF5350' for v in imps], edgecolor='white')
ax.set_ylabel('Improvement (%)'); ax.set_title('Accuracy Gain from Confidence Filter')
ax.set_xticklabels(names, rotation=30, ha='right', fontsize=8)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved: model_comparison_chart.png')

---
## 7 — Summary

All models evaluated with **confidence margin = ±0.10**.

In [ ]:
print('Done. All models evaluated with confidence margin = ±0.10')